## Hands on real clustering data - DESI DR1 LRG sample
In this session we will estimate (and interpret) the correlation function of galaxy catalogs.
This is the first (compression) step of a standard clustering analysis; the second step consists in fitting these compressed measurements with a theory model to derive constraints on cosmological parameters.

### Installation

In [ ]:
!pip install git+https://github.com/cosmodesi/cosmoprimo#egg=cosmoprimo[camb]

#### Catalogs
Let's download DESI DR1 LRG catalogs available here: https://data.desi.lbl.gov/doc/releases/dr1/

In [ ]:
!wget https://data.desi.lbl.gov/public/dr1/survey/catalogs/dr1/LSS/iron/LSScats/v1.5/LRG_NGC_clustering.dat.fits
!wget https://data.desi.lbl.gov/public/dr1/survey/catalogs/dr1/LSS/iron/LSScats/v1.5/LRG_NGC_0_clustering.ran.fits

## Inspecting catalogs

In [ ]:
path_data = "LRG_NGC_clustering.dat.fits"
path_randoms = "LRG_NGC_0_clustering.ran.fits"

import numpy as np

from astropy.table import Table

data = Table.read(path_data)
randoms = Table.read(path_randoms)
print('Columns of the data (galaxy) catalog', list(data.columns))
print('Columns of the catalog of randoms', list(randoms.columns))

### RA/Dec distribution

In [ ]:
# Make a scatter plot of data RA/Dec and a histogram of data redshifts
# Check that weighted randoms have the same angular (RA/Dec) and redshift (Z) distribution as the data
from matplotlib import pyplot as plt
# Tip: for the RA/Dec plot, downsample the data and randoms for faster plots:
rng = np.random.RandomState(seed=42)
mask_data = rng.uniform(0., 1., len(data)) < 0.1
mask_randoms = rng.uniform(0., 1., len(randoms)) < mask_data.sum() / len(randoms)
# Then, e.g. plt.scatter(data['RA'][mask_data], data['DEC'][mask_data], s=1, label='data')
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.scatter(randoms['RA'][mask_randoms], randoms['DEC'][mask_randoms], s=1, label='randoms', color='C0')
ax.scatter(data['RA'][mask_data], data['DEC'][mask_data], s=1, label='data', color='C1')
ax.legend(frameon=False)
ax.set_xlabel('R.A. [deg]')
ax.set_ylabel('Dec. [deg]')
plt.show()

### Redshift ($z$) distribution

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.hist(data['Z'], histtype='step', density=True, label='data', color='C1')
ax.hist(randoms['Z'], histtype='step', density=True, label='randoms', color='C0')
ax.legend(frameon=False)
ax.set_xlabel('$z$')
plt.show()

# In the following, we will select a subsample of data, 0.4 < z < 0.6
zlim = (0.4, 0.6)
mask_zdata = (data['Z'] > zlim[0]) & (data['Z'] < zlim[1])
mask_zrandoms = (randoms['Z'] > zlim[0]) & (randoms['Z'] < zlim[1])

Let's first transform redshifts $z$ into distances, assuming a fiducial cosmology. Take as a fiducial cosmology the best fit cosmology of Planck2018 (TT, TE, EE, lowl, lensing).

In [ ]:
from cosmoprimo.fiducial import DESI
cosmo_fid = DESI(engine='camb')  # fiducial cosmology

def get_cartesian_positions(ra, dec, z):
    # Compute distance d
    d = cosmo_fid.comoving_radial_distance(z)
    # Turn distance d, RA (\phi), Dec (\pi/2-\theta) (mind degree -> radians!) into x, y, z Cartesian positions
    conv = np.pi / 180.
    theta, phi = dec * conv, ra * conv
    # TODO: Fill in x, y, z positions:
    # x = ...
    # y = ...
    # z = ...
    return np.column_stack([x, y, z])

data_positions = get_cartesian_positions(data['RA'], data['DEC'], data['Z'])
# Same for randoms
randoms_positions = get_cartesian_positions(randoms['RA'], randoms['DEC'], randoms['Z'])

For fun, make a 'wedge plot' of the data: a Cartesian 2D (x, y) slice between 1 and 2 deg in Dec. Do you see structures, filaments, voids?

In [ ]:
lim_dec = (1., 2.)
mask_data = (data['DEC'] > lim_dec[0]) & (data['DEC'] < lim_dec[1])
mask_randoms = (randoms['DEC'] > lim_dec[0]) & (randoms['DEC'] < lim_dec[1])

fig, ax = plt.subplots(1, 1, figsize=(10,) * 2)
ax.set_aspect('equal')
# Plot (-x, y) (minus sign just to orient the figure)
ax.scatter(- randoms_positions[mask_randoms, 0], randoms_positions[mask_randoms, 1], marker='.', s=40., alpha=0.01, color='C0')
ax.scatter(- data_positions[mask_data, 0], data_positions[mask_data, 1], marker='.', s=0.3, alpha=0.8, color='C1')
ax.tick_params(bottom=False, labelbottom=False, left=False, labelleft=False)
plt.show()

Galaxies in the data and randoms catalogs receive weights, to correct for observational systematic effects, such that the ensemble average of galaxy density (= "survey selection function") and that of randoms match:
- WEIGHT_SYS: weights to correct for photometric systematics: what are they? **Answer**: non-cosmological variations of target density (due to e.g. galactic dust, varying atmospheric, sky conditions)
- WEIGHT_COMP: weights to correct for fiber collisions: what are they? **Answer**: all galaxies do not receive a spectroscopic fibers (due to the limited density of fibers)
- WEIGHT_ZFAIL: weights to correct for redshift failures: what are they? **Answer**: reliable redshifts are not obtained for all galaxies (due to instrumental and observational conditions)

The total (completeness) weight is (up to some renormalization): WEIGHT = WEIGHT_SYS * WEIGHT_COMP * WEIGHT_ZFAIL.

In addition to completeness weights above, when computing 2pt statistics (correlation function or power spectrum), one can apply weights to minimize its variance: WEIGHT_FKP = 1/(1 + NZ * P0), with P0 the typical value of the power spectrum at the scales of interest, e.g. $10000 \; (\mathrm{Mpc}/h)^{3}$ (NZ is in $(\mathrm{Mpc}/h)^{-3}$).
See e.g. https://arxiv.org/pdf/astro-ph/9304022.pdf, eq. 2.3, for the variational demonstration (another, broader point-of-view is that of the optimal quadratic estimator, of which the FKP estimator we will use below is a simplification under some assumptions).

In [ ]:
# Compute data weights, WEIGHT * WEIGHT_FKP
data_weights = data['WEIGHT'] * data['WEIGHT_FKP']
# Same for randoms
randoms_weights = randoms['WEIGHT'] * randoms['WEIGHT_FKP']

# Now let's apply the z-selection
data_positions, data_weights = data_positions[mask_zdata], data_weights[mask_zdata]
randoms_positions, randoms_weights = randoms_positions[mask_zrandoms], randoms_weights[mask_zrandoms]

## Correlation function

Correlation functions are usually estimated by binning the (weighted) number of pairs of particles (galaxies, randoms) as a function of the distance $s = |\mathbf{r}_2 - \mathbf{r}_1|$ between particles (and optionally the cosine angle to the line-of-sight $\mu = \hat{\mathbf{s}} \cdot \widehat{\mathbf{r}_2 + \mathbf{r}_1} = \mathbf{s} \cdot (\mathbf{r}_2 + \mathbf{r}_1) / (s |\mathbf{r}_1 + \mathbf{r}_2|))$.

This is basically a double loop, schematically:
```
for i1 in range(n1):  # loop over particles (positions, weights) of catalog 1
    for i2 in range(n2):  # loop over particles (positions, weights) of catalog 2
        wcounts[index(s(i1, i2)), index(mu(i1, i2))] += weight(i1, i2)
```
See also the notebook "paris_stations.ipynb".
If you have some time, you can give it a try with Python / numba; on Google Colab to start with you can e.g. ask the AI for "compute number of pair of points as a function of separation in 3D and cosine angle mu of the pair with respect to the midpoint line of sight, accelerate with numba
" (NOTE: be careful! the code may not be correct!). Compute the pair counts of galaxies ```data_positions1``` with weights ```data_weights1```, $s$-edges ```np.linspace(0., 160, 41)``` and $\mu$-edges ```np.linspace(-1., 1., 100)```. Downsample ```data_positions``` by a factor ~40 for the test --- how long without downsampling?

In [ ]:
import numba as nb

@nb.njit(parallel=True)
def get_pair_counts(sedges, muedges, positions1, weights1, positions2, weights2):
    """
    Compute number of pair of points as a function of separation in 3D and cosine angle mu of the pair with respect to the midpoint line of sight, accelerate with numba.

    Parameters
    ----------
    sedges : array of shape (Ns + 1, )
        Bin edges for the distance.
    muedges : array of shape (Nmu + 1, )
        Bin edges for the cosine angle.
    positions1 : array of shape (N1, 3)
        Cartesian positions of points in the first catalog.
    weights1 : array of shape (N1, )
        Weights of points in the first catalog.
    positions2 : array of shape (N2, 3)
        Cartesian positions of points in the second catalog.
    weights2 : array of shape (N2, )
        Weights of points in the second catalog.

    Returns
    -------
    counts : array of shape (Ns, Nmu)
        Number of pairs in each separation bin.
    """
    n1 = positions1.shape[0]
    n2 = positions2.shape[0]
    wcounts = np.zeros((sedges.shape[0] - 1, muedges.shape[0] - 1), dtype='f8')

    for i1 in nb.prange(n1):
        for i2 in range(n2):
            diff = positions2[i2] - positions1[i1]
            s2 = np.sum(diff**2)
            if s2 > sedges[-1]**2 or s2 <= sedges[0]**2: continue
            s = np.sqrt(s2)
            los = (positions1[i1] + positions2[i2])
            mu = np.sum(diff * los) / (s * np.sqrt(np.sum(los**2)))
            # Compute sindex, muindex and add weight = weights1[i1] * weights2[i2]
            #wcounts[sindex, muindex] += ...

    return wcounts

In [ ]:
%%time
sedges = np.linspace(0., 160, 41)
muedges = np.linspace(-1., 1., 100)  # s, mu binning

# Downsample by a factor 40
sl = slice(0, None, 40)
DD_counts = get_pair_counts(sedges, muedges, data_positions[sl], data_weights[sl], data_positions[sl], data_weights[sl])

In [ ]:
# Plot pair counts
sep = (sedges[:-1] + sedges[1:]) / 2.
counts = np.sum(DD_counts, axis=-1)
fig, ax = plt.subplots()
ax.plot(sep, counts / (sedges[1:]**3 - sedges[:-1]**3))
ax.set_xscale('log')
ax.set_xlabel('$s$ [Mpc/$h$]')
ax.set_ylabel('Pair counts')
plt.show()

Now let's move to a faster pair counter!

In [ ]:
pip install git+https://github.com/adematti/cucount.git

In [ ]:
sedges = np.linspace(0., 160, 41)
muedges = np.linspace(-1., 1., 100)  # s, mu binning

# Just for this notebook, let's further downsample randoms to 3x times the size of the data
mask_randoms = rng.uniform(0., 1., randoms_positions.shape[0]) < 3 * data_positions.shape[0] / randoms_positions.shape[0]
randoms_positions, randoms_weights = randoms_positions[mask_randoms], randoms_weights[mask_randoms]

los = 'midpoint'

from cucount.numpy import count2, Particles, BinAttrs, setup_logging

# Compute pair counts
data_particles = Particles(data_positions, data_weights)
randoms_particles = Particles(randoms_positions, randoms_weights)
battrs = BinAttrs(s=sedges, mu=(muedges, los))

In [ ]:
%%time
DD = count2(data_particles, data_particles, battrs=battrs)['weight']

In [ ]:
%%time
DR = count2(data_particles, randoms_particles, battrs=battrs)['weight']
RR = count2(randoms_particles, randoms_particles, battrs=battrs)['weight']

In [ ]:
# Compute the normalization of pair counts = total weighted number of pairs
DD_norm = np.sum(data_weights)**2 - np.sum(data_weights**2)
DR_norm = np.sum(data_weights) * np.sum(randoms_weights)
RR_norm = np.sum(randoms_weights)**2 - np.sum(randoms_weights**2)

# Landy-Szalay estimator, (DD - 2DR + RR) / RR
xi = ((DD / DD_norm) - 2 * (DR / DR_norm) + (RR / RR_norm)) / (RR / RR_norm)

### Multipoles of the correlation function
Compute and plot the correlation function multipoles $\ell = 0$ (monopole), $\ell = 2$ (quadrupole) and $\ell = 4$ (hexadecapole), given by $\xi_{\ell}(s) = \frac{2 \ell + 1}{2} \int_{-1}^{1} d\mu \xi(s,\mu) \mathcal{L}_{\ell}(\mu)$,
with $\mathcal{L}_{\ell}(\mu)$ Legendre polynomials (see https://en.wikipedia.org/wiki/Legendre_polynomials, also https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.legendre.html).

In [ ]:
# Trick: write d\mu \mathcal{L}_{\ell}(\mu) as the difference of \mathcal{L}_{\ell}(\mu) primitives
# This makes sure $\xi_{\ell > 0} is 0 in case $\xi(s, \mu)$ is constant in \mu
ells = (0, 2, 4)
# Compute xiell, list of multipoles
from scipy import special
xiell = []
for ell in ells:
    poly = special.legendre(ell).integ()(muedges)
    legendre = (poly[1:] - poly[:-1])
    xiell.append(np.sum((2 * ell + 1) * xi * legendre, axis=-1) / (muedges[-1] - muedges[0]))

# Plot the correlation function multipoles
fig, ax = plt.subplots()
for ill, ell in enumerate(ells):
    ax.plot(sep, sep**2 * xiell[ill], label=rf'$\ell = {ell:d}$')
ax.grid(True)
ax.set_xlabel(r'$s$ [$\mathrm{Mpc}/h$]')
ax.set_ylabel(r'$s^{2}\xi_{\ell}(s)$ [$(\mathrm{Mpc}/h)^{2}$]')
ax.legend(frameon=False)
plt.show()